In [ ]:
import astropy.units as u
import astropy.constants as const
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from tqdm.notebook import tqdm
import json
import h5py
import os

import warnings
warnings.filterwarnings("ignore")

from DiscEvolution.grid import Grid
from DiscEvolution.star import SimpleStar
from DiscEvolution.eos import LocallyIsothermalEOS  
from DiscEvolution.eos import SimpleDiscEOS
from DiscEvolution.eos import IrradiatedEOS  
from DiscEvolution.disc import AccretionDisc
from DiscEvolution.viscous_evolution import ViscousEvolutionFV
from DiscEvolution.driver import PlanetDiscDriver
from DiscEvolution.io import Event_Controller
from DiscEvolution.dust import DustGrowthTwoPop, SingleFluidDrift
from DiscEvolution.viscous_evolution import HybridWindModel
from DiscEvolution.opacity import Zhu2012
from DiscEvolution.constants import yr as yr_code, GasConst, Omega0, AU, Msun  # DiscEvolution uses code units: 1 year = 2π code time units
from DiscEvolution.chemistry import SimpleCOAtomAbund, EquilibriumCOChemOberg
from DiscEvolution import planet_formation as pf
from DiscEvolution.diffusion import TracerDiffusion 
from DiscEvolution.planet_formation import Planets, Bitsch2015Model
from DiscEvolution.constants import *

In [ ]:
def h5_to_dict(h5obj):
    out = {}
    for k, v in h5obj.items():
        if isinstance(v, h5py.Dataset):
            out[k] = v[()]          # load dataset into memory
        else:
            out[k] = h5_to_dict(v)  # recurse into group
    return out     

crPath = "/Users/james/Documents/GitHub/DiscEvolution/james/august2026/output/CR_0.01Myr/trig/winds_mig_psi10.0_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260901_170258.h5"

print('\n  -------- data ---------')
with h5py.File(crPath, "r") as f:
    CRdata = h5_to_dict(f)
    CRattrs = dict(f.attrs)

for i in CRdata.keys():
    print(i)



  -------- data ---------
B_xe
C_xe
D_xe
H
JXR
Lambda
M_iso
M_transition
Mcs
Mes
NH
P
R
R_dz
R_dz_t
Rp
Sigc
Sigma_G
Sigma_dust
Sigma_grain_size
Sigma_pebble_size
Sigma_pebbles
Sigma_planetesimals
St
T
Tc
Vdrift
X_cores
X_envs
alpha_R
disk_Mass
disk_Mdot_p
disk_Mdot_star
disk_atom_gas_abund
disk_atom_ice_abund
disk_ice_lines
disk_mol_gas_abund
disk_mol_ice_abund
disk_planetesimal_atom_abund
disk_planetesimal_mol_abund
eta_diff
ice_lines
n_density
nu
psi_R
rho_mid
sigmaXR
t
tau_XR
time_snap
xe
zeta
zeta_CR
zeta_XR


In [3]:
crR = CRdata["R"]
crtsnap = CRdata["time_snap"]    # Myr        
crSigma_g = CRdata["Sigma_G"]   

crRdz = CRdata['R_dz_t']
crt = CRdata["t"]                # yr

crZeta = CRdata['zeta']
crZetaCR = CRdata['zeta_CR']
crZetaXR = CRdata['zeta_XR']

crLambda = CRdata['Lambda']
crT = CRdata['T']
crxe = CRdata['xe']